# State-based History-aware Artificial Reinforcement Interpretable Kernel" (Sharik)
## Open AI Gym - Atari - Breakout - Experiential Learning - Exploring Extended Hyper-Parameters  

- $mc$ --motivated_curiosity, type=float, default=0.0 - Motivated (by surprise) curiosity
  - $c_t$ chance to perform random action ("curiosity"), dependant on "surprizingness" $z_t = norm\_sim(s_t,s'_t)$, as $c_t = mc * z_t$
- $cc$ --constant curiosity (inverse to greediness) - chance to perform random action regardless of the "surprizingness"  
- $er$ --expectedness_reward, type=float, default=0.0 - Expectedness reward
  - $g_t$ - extra positive feedback for predictiviness $1 - z_t$, as $g_t = er * (1-z_t)$, where $er$ can be considered as an element of agent's personal profile $x$
- $sr$ --state_reward, type=int, default=1 - Place reward in state (1) or not (0)
  
So far, none of them works - for seeds 2, 3, and 41 the best results are still for $er=0, mc=0, cc=0, sr=1$


In [1]:
import os, sys
import pickle
import pandas as pd
import queue
import matplotlib.pyplot as plt
import random
import math
import numpy as np

## Experiment with Motivated (Conditioned) Curiosity and Expectedness (Predictiveness) Reward based on Surprisingness (anti-Predictiveness)

In [19]:
# er, mc: 1M

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=0.5 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er05mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=1.0 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er10mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=0.5 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=1.0 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc10.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=0.5 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er05mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=1.0 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er10mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=0.5 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=1.0 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc10.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=0.5 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er05mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -er=1.0 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er10mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=0.5 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sm="exp(-d)" -mc=1.0 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc10.txt

#=>

#er0mc0 => 8.83 (BEST!)
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:08.072909"; time="0:19:32.297677"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:02.481580"; time="0:19:42.842152"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.756226"; time="0:19:42.002109"

#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:08.072909"; time="0:19:32.297677"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc05.txt:score_avg=4.3; steps_tot=1080000; steps_avg=1118.0; lives_avg=0.0; lapse_avg="0:00:00.570908"; time="0:08:58.610062"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er0mc10.txt:score_avg=5.5; steps_tot=1080000; steps_avg=1399.0; lives_avg=0.0; lapse_avg="0:00:00.194261"; time="0:22:43.029088"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er05mc0.txt:score_avg=0.2; steps_tot=1080000; steps_avg=563.4; lives_avg=0.0; lapse_avg="0:00:00.242842"; time="0:10:05.387658"
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=1080000; steps_avg=847.1; lives_avg=0.0; lapse_avg="0:00:00.069858"; time="0:10:36.342916"

#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:02.481580"; time="0:19:42.842152"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc05.txt:score_avg=3.6; steps_tot=1080000; steps_avg=1609.5; lives_avg=0.1; lapse_avg="0:00:01.505553"; time="0:08:47.056933"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er0mc10.txt:score_avg=3.5; steps_tot=1080000; steps_avg=1005.6; lives_avg=0.0; lapse_avg="0:00:00.057235"; time="0:22:46.613459"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er05mc0.txt:score_avg=3.0; steps_tot=1080000; steps_avg=915.3; lives_avg=0.0; lapse_avg="0:00:00.343216"; time="0:10:05.306984"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0er10mc0.txt:score_avg=3.0; steps_tot=1080000; steps_avg=1013.1; lives_avg=0.0; lapse_avg="0:00:00.390394"; time="0:10:37.310238"

#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.756226"; time="0:19:42.002109"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc05.txt:score_avg=6.0; steps_tot=1080000; steps_avg=1438.1; lives_avg=0.0; lapse_avg="0:00:00.030599"; time="0:08:51.614975"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er0mc10.txt:score_avg=4.1; steps_tot=1080000; steps_avg=1193.4; lives_avg=0.0; lapse_avg="0:00:00.559716"; time="0:22:47.393618"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er05mc0.txt:score_avg=1.0; steps_tot=1080000; steps_avg=726.3; lives_avg=0.0; lapse_avg="0:00:00.127887"; time="0:10:06.467727"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=1080000; steps_avg=716.2; lives_avg=0.0; lapse_avg="0:00:00.014154"; time="0:10:38.636689"


# er, mc: 5M
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)"         > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er0mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -er=0.5 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er05mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -er=1.0 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er10mc0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -mc=0.5 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er0mc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -mc=1.0 > sim_test4/exp$2_s$1ss10lm2sc2cs2tu0er0mc10.txt

#=>

#(env) akolonin@users-MacBook-Pro aigents-python % grep avg sim_test4/exp5400000_s*er*mc*.txt

#er0mc0 => 18.87 (BEST!)
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=22.2; steps_tot=5400000; steps_avg=4048.0; lives_avg=0.2; lapse_avg="0:00:01.981214"; time="1:18:23.923148"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=14.0; steps_tot=5400000; steps_avg=2880.0; lives_avg=0.1; lapse_avg="0:00:07.345784"; time="1:19:26.060987"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=20.4; steps_tot=5400000; steps_avg=2825.7; lives_avg=0.0; lapse_avg="0:00:00.402703"; time="1:19:57.444630"

#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er05mc0.txt:score_avg=0.2; steps_tot=5400000; steps_avg=541.8; lives_avg=0.0; lapse_avg="0:00:00.347051"; time="1:04:01.298912"
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er0mc05.txt:score_avg=6.8; steps_tot=5400000; steps_avg=1419.9; lives_avg=0.0; lapse_avg="0:00:00.181880"; time="10:09:17.618253"
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er0mc10.txt:score_avg=7.5; steps_tot=5400000; steps_avg=1815.7; lives_avg=0.0; lapse_avg="0:00:00.080085"; time="2:12:08.482641"
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=5400000; steps_avg=712.3; lives_avg=0.0; lapse_avg="0:00:00.223011"; time="0:48:21.687585"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er05mc0.txt:score_avg=3.0; steps_tot=5400000; steps_avg=906.2; lives_avg=0.0; lapse_avg="0:00:00.205336"; time="1:03:52.888159"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er0mc05.txt:score_avg=5.9; steps_tot=5400000; steps_avg=2222.2; lives_avg=0.1; lapse_avg="0:00:06.601391"; time="10:08:45.366491"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er0mc10.txt:score_avg=5.3; steps_tot=5400000; steps_avg=1251.2; lives_avg=0.0; lapse_avg="0:00:00.122420"; time="2:12:38.384006"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0er10mc0.txt:score_avg=3.0; steps_tot=5400000; steps_avg=989.7; lives_avg=0.0; lapse_avg="0:00:00.393639"; time="0:47:55.091749"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er05mc0.txt:score_avg=1.0; steps_tot=5400000; steps_avg=688.6; lives_avg=0.0; lapse_avg="0:00:00.209414"; time="1:03:58.965975"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er0mc05.txt:score_avg=8.7; steps_tot=5400000; steps_avg=1747.6; lives_avg=0.0; lapse_avg="0:00:01.194638"; time="9:05:50.786622"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er0mc10.txt:score_avg=7.1; steps_tot=5400000; steps_avg=1563.0; lives_avg=0.0; lapse_avg="0:00:00.135491"; time="2:13:04.690199"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0er10mc0.txt:score_avg=1.0; steps_tot=5400000; steps_avg=688.6; lives_avg=0.0; lapse_avg="0:00:00.482769"; time="1:36:08.487472"



## Experiment with State Reward (Reward in State)  

In [27]:
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sr=1 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sr=1 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41  -mg=5000000 -mt=1080000 -sr=1 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr1.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=1080000 -sr=0 > sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=1080000 -sr=0 > sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=1080000 -sr=0 > sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr0.txt

# sr1 => 8.83 (BEST!)
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr1.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:08.468077"; time="0:11:17.523051"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr1.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:02.342758"; time="0:11:31.564646"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr1.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.757189"; time="0:14:26.142090"

# sr0 => 6.5
#sim_test4/exp1080000_s2ss10lm2sc2cs2tu0sr0.txt:score_avg=4.1; steps_tot=1080000; steps_avg=1483.5; lives_avg=0.1; lapse_avg="0:00:00.875952"; time="0:11:35.934361"
#sim_test4/exp1080000_s3ss10lm2sc2cs2tu0sr0.txt:score_avg=9.5; steps_tot=1080000; steps_avg=2080.9; lives_avg=0.1; lapse_avg="0:00:01.080624"; time="0:11:32.168979"
#sim_test4/exp1080000_s41ss10lm2sc2cs2tu0sr0.txt:score_avg=6.0; steps_tot=1080000; steps_avg=2926.8; lives_avg=0.2; lapse_avg="0:00:19.326955"; time="0:14:25.349930"

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=5400000 -sr=1 > sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=5400000 -sr=1 > sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr1.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41  -mg=5000000 -mt=5400000 -sr=1 > sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr1.txt

#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=2  -mg=5000000 -mt=5400000 -sr=0 > sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=3  -mg=5000000 -mt=5400000 -sr=0 > sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr0.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -tu=0 -s=41 -mg=5000000 -mt=5400000 -sr=0 > sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr0.txt

# sr1 => 18.87 (BEST!)
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr1.txt:score_avg=22.2; steps_tot=5400000; steps_avg=4048.0; lives_avg=0.2; lapse_avg="0:00:01.262753"; time="1:03:12.349516"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr1.txt:score_avg=14.0; steps_tot=5400000; steps_avg=2880.0; lives_avg=0.1; lapse_avg="0:00:04.120212"; time="1:03:21.890194"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr1.txt:score_avg=20.4; steps_tot=5400000; steps_avg=2825.7; lives_avg=0.0; lapse_avg="0:00:00.196652"; time="1:03:20.204869"

# sr0 => 12.47
#sim_test4/exp5400000_s2ss10lm2sc2cs2tu0sr0.txt:score_avg=6.9; steps_tot=5400000; steps_avg=1706.2; lives_avg=0.0; lapse_avg="0:00:00.046152"; time="0:59:36.553148"
#sim_test4/exp5400000_s3ss10lm2sc2cs2tu0sr0.txt:score_avg=19.5; steps_tot=5400000; steps_avg=3184.0; lives_avg=0.1; lapse_avg="0:00:00.625007"; time="0:59:15.546036"
#sim_test4/exp5400000_s41ss10lm2sc2cs2tu0sr0.txt:score_avg=11.0; steps_tot=5400000; steps_avg=3688.5; lives_avg=0.3; lapse_avg="0:00:00.413047"; time="0:59:01.144719"



## Experiment with Space Encodings and Transition Utility  

In [30]:
#mt=1080000

#se=lr tu=-1 mt=1080000 => 6.9
#grep avg sim_test4/lr_exp1*_s*tu-1*
#sim_test4/lr_exp1080000_s2ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=2.9; steps_tot=1080000; steps_avg=897.0; lives_avg=0.0; lapse_avg="0:00:00.461690"; time="0:24:02.677944"
#sim_test4/lr_exp1080000_s3ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=16.2; steps_tot=1080000; steps_avg=2482.8; lives_avg=0.0; lapse_avg="0:00:01.133845"; time="0:23:49.070871"
#sim_test4/lr_exp1080000_s41ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=1.6; steps_tot=1080000; steps_avg=737.2; lives_avg=0.0; lapse_avg="0:00:00.346554"; time="0:24:03.563048"

#se=na tu=-1 mt=1080000 => 0.52
#grep avg sim_test4/na_exp1*_s*tu-1*
#sim_test4/na_exp1080000_s2ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=1.6; steps_tot=1080000; steps_avg=708.7; lives_avg=0.0; lapse_avg="0:00:00.236803"; time="0:31:07.726632"
#sim_test4/na_exp1080000_s3ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=0.0; steps_tot=1080000; steps_avg=532.8; lives_avg=0.0; lapse_avg="0:00:00.132575"; time="0:16:42.130917"
#sim_test4/na_exp1080000_s41ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=0.0; steps_tot=1080000; steps_avg=487.8; lives_avg=0.0; lapse_avg="0:00:00.055094"; time="0:31:11.148001"

#se=lr tu=0 mt=1080000 => 5.5
#grep avg sim_test4/lr_exp1*_s*tu0* 
#sim_test4/lr_exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=7.9; steps_tot=1080000; steps_avg=1406.2; lives_avg=0.0; lapse_avg="0:00:01.066499"; time="0:12:18.062631"
#sim_test4/lr_exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=6.6; steps_tot=1080000; steps_avg=1391.8; lives_avg=0.0; lapse_avg="0:00:01.193517"; time="0:12:19.106639"
#sim_test4/lr_exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=2.1; steps_tot=1080000; steps_avg=817.6; lives_avg=0.0; lapse_avg="0:00:00.021099"; time="0:12:22.945913"

#se=na tu=0 mt=1080000 => 8.83 (BEST!)
#grep avg sim_test4/na_exp1*_s*tu0*
#sim_test4/na_exp1080000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=8.0; steps_tot=1080000; steps_avg=2693.3; lives_avg=0.2; lapse_avg="0:00:07.856224"; time="0:11:25.118635"
#sim_test4/na_exp1080000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=7.2; steps_tot=1080000; steps_avg=1767.6; lives_avg=0.0; lapse_avg="0:00:02.194325"; time="0:11:31.607772"
#sim_test4/na_exp1080000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=11.3; steps_tot=1080000; steps_avg=2049.3; lives_avg=0.0; lapse_avg="0:00:00.591159"; time="0:11:27.631181"

#mt=5400000

#se=lr tu=-1 mt=5400000 => 7.1 (STUCK!)
#grep avg sim_test4/lr_exp5*_s*tu-1*
#sim_test4/lr_exp5400000_s2ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=2.9; steps_tot=5400000; steps_avg=886.3; lives_avg=0.0; lapse_avg="0:00:00.167112"; time="4:07:11.552640"
#sim_test4/lr_exp5400000_s3ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=16.8; steps_tot=5400000; steps_avg=2522.2; lives_avg=0.0; lapse_avg="0:00:00.268579"; time="4:06:39.367544"
#sim_test4/lr_exp5400000_s41ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=1.6; steps_tot=5400000; steps_avg=731.8; lives_avg=0.0; lapse_avg="0:00:00.282968"; time="4:07:03.540383"

#se=na tu=-1 mt=5400000 => 0.17
#grep avg sim_test4/na_exp5*_s*tu-1*
#sim_test4/na_exp5400000_s2ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=0.5; steps_tot=5400000; steps_avg=555.8; lives_avg=0.0; lapse_avg="0:00:00.239207"; time="1:57:10.931704"
#sim_test4/na_exp5400000_s3ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=0.0; steps_tot=5400000; steps_avg=501.3; lives_avg=0.0; lapse_avg="0:00:00.156470"; time="1:56:13.200245"
#sim_test4/na_exp5400000_s41ss10lm2sc2cs2tu-1er0mc0.txt:score_avg=0.0; steps_tot=5400000; steps_avg=487.3; lives_avg=0.0; lapse_avg="0:00:00.388070"; time="1:57:32.548924"

#se=lr tu=0 mt=5400000 => 6.4
#sim_test4/lr_exp5400000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=10.1; steps_tot=5400000; steps_avg=1520.7; lives_avg=0.0; lapse_avg="0:00:00.381913"; time="1:05:02.977835"
#sim_test4/lr_exp5400000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=7.2; steps_tot=5400000; steps_avg=1481.9; lives_avg=0.0; lapse_avg="0:00:00.544621"; time="1:05:34.013855"
#sim_test4/lr_exp5400000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=1.9; steps_tot=5400000; steps_avg=803.7; lives_avg=0.0; lapse_avg="0:00:00.313763"; time="1:05:01.940513"

#se=na tu=0 mt=5400000 => 18.87 (BEST!)
#grep avg sim_test4/na_exp5*_s*tu0*
#sim_test4/na_exp5400000_s2ss10lm2sc2cs2tu0er0mc0.txt:score_avg=22.2; steps_tot=5400000; steps_avg=4048.0; lives_avg=0.2; lapse_avg="0:00:01.545458"; time="1:00:37.752103"
#sim_test4/na_exp5400000_s3ss10lm2sc2cs2tu0er0mc0.txt:score_avg=14.0; steps_tot=5400000; steps_avg=2880.0; lives_avg=0.1; lapse_avg="0:00:04.838446"; time="1:00:24.810818"
#sim_test4/na_exp5400000_s41ss10lm2sc2cs2tu0er0mc0.txt:score_avg=20.4; steps_tot=5400000; steps_avg=2825.7; lives_avg=0.0; lapse_avg="0:00:00.320971"; time="1:00:50.475942"


## Experiment with Constant/Motivated Curiosity and Transition Utility  

In [35]:
#grep avg sim_test4/*_exp1*_s*ss10lm2sc2cs2tu*er0mc0cc01.txt

#mt=1080000s

#se=lr tu=-1 cc=0.1 mt=1080000 => 10.23 (BEST!)
#sim_test4/lr_exp1080000_s2ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=4.7; steps_tot=1080000; steps_avg=1001.9; lives_avg=0.0; lapse_avg="0:00:00.334675"; time="0:13:18.181131"
#sim_test4/lr_exp1080000_s3ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=23.0; steps_tot=1080000; steps_avg=2903.2; lives_avg=0.0; lapse_avg="0:00:01.632634"; time="0:13:18.931470"
#sim_test4/lr_exp1080000_s41ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=3.0; steps_tot=1080000; steps_avg=869.6; lives_avg=0.0; lapse_avg="0:00:00.416042"; time="0:13:17.669154"

#se=na tu=-1 cc=0.1 mt=1080000 => 1.8
#sim_test4/na_exp1080000_s2ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=1.7; steps_tot=1080000; steps_avg=763.3; lives_avg=0.0; lapse_avg="0:00:00.344160"; time="0:10:07.483130"
#sim_test4/na_exp1080000_s3ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=1.8; steps_tot=1080000; steps_avg=777.5; lives_avg=0.0; lapse_avg="0:00:00.240272"; time="0:10:07.605687"
#sim_test4/na_exp1080000_s41ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=1.9; steps_tot=1080000; steps_avg=780.3; lives_avg=0.0; lapse_avg="0:00:00.400964"; time="0:10:15.907209"

#se=lr tu=0 cc=0.1 mt=1080000 => 3.8
#sim_test4/lr_exp1080000_s2ss10lm2sc2cs2tu0er0mc0cc01.txt:score_avg=12.4; steps_tot=1080000; steps_avg=1985.3; lives_avg=0.0; lapse_avg="0:00:00.097228"; time="3:22:36.916696"
#sim_test4/lr_exp1080000_s3ss10lm2sc2cs2tu0er0mc0cc01.txt:score_avg=5.0; steps_tot=1080000; steps_avg=1188.1; lives_avg=0.0; lapse_avg="0:00:00.075415"; time="3:22:50.083421"
#sim_test4/lr_exp1080000_s41ss10lm2sc2cs2tu0er0mc0cc01.txt:score_avg=4.0; steps_tot=1080000; steps_avg=1038.5; lives_avg=0.0; lapse_avg="0:00:00.169906"; time="3:22:44.573530"

#se=na tu=0 cc=0.1 mt=1080000 => 2.5
#sim_test4/na_exp1080000_s2ss10lm2sc2cs2tu0er0mc0cc01.txt:score_avg=2.6; steps_tot=1080000; steps_avg=894.0; lives_avg=0.0; lapse_avg="0:00:00.597933"; time="0:12:15.719881"
#sim_test4/na_exp1080000_s3ss10lm2sc2cs2tu0er0mc0cc01.txt:score_avg=2.5; steps_tot=1080000; steps_avg=883.1; lives_avg=0.0; lapse_avg="0:00:00.518785"; time="0:12:14.028703"
#sim_test4/na_exp1080000_s41ss10lm2sc2cs2tu0er0mc0cc01.txt:score_avg=2.5; steps_tot=1080000; steps_avg=886.0; lives_avg=0.0; lapse_avg="0:00:00.330810"; time="0:12:07.597388"

#TODO 540000000 - tu-1 cc=0.1 
# DOING...

#se=lr tu=-1 cc=0.1 mt=5400000 => 10.7 (STUCK!)
#sim_test4/lr_exp5400000_s2ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=4.8; steps_tot=5400000; steps_avg=1012.2; lives_avg=0.0; lapse_avg="0:00:00.708348"; time="1:06:38.009115"
#sim_test4/lr_exp5400000_s3ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=24.4; steps_tot=5400000; steps_avg=3020.1; lives_avg=0.0; lapse_avg="0:00:00.727375"; time="1:06:35.297158"
#sim_test4/lr_exp5400000_s41ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=3.0; steps_tot=5400000; steps_avg=875.5; lives_avg=0.0; lapse_avg="0:00:00.209006"; time="1:06:54.434082"

#se=na tu=-1 cc=0.1 mt=5400000 => 2.7
#sim_test4/na_exp5400000_s2ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=2.7; steps_tot=5400000; steps_avg=906.8; lives_avg=0.0; lapse_avg="0:00:00.404097"; time="0:59:17.956625"
#sim_test4/na_exp5400000_s3ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=2.6; steps_tot=5400000; steps_avg=904.1; lives_avg=0.0; lapse_avg="0:00:00.215278"; time="0:59:36.732136"
#sim_test4/na_exp5400000_s41ss10lm2sc2cs2tu-1er0mc0cc01.txt:score_avg=2.8; steps_tot=5400000; steps_avg=913.9; lives_avg=0.0; lapse_avg="0:00:00.390778"; time="0:59:20.272994"

#...

#...



#TODO cc = 0.5

#DOING...
#5400000
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -tu=-1 -cc=0.5 -se="leftright" > sim_test4/lr_exp$2_s$1ss10lm2sc2cs2tu-1er0mc0cc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -tu=-1 -cc=0.5 -se="na"        > sim_test4/na_exp$2_s$1ss10lm2sc2cs2tu-1er0mc0cc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -tu=0  -cc=0.5 -se="leftright" > sim_test4/lr_exp$2_s$1ss10lm2sc2cs2tu0er0mc0cc05.txt
#python ./aigents-gym/breakout_eval2.py -cs=2 -ss=1.0 -s=$1 -mg=50000000 -mt=$2 -sm="exp(-d)" -tu=0  -cc=0.5 -se="na"        > sim_test4/na_exp$2_s$1ss10lm2sc2cs2tu0er0mc0cc05.txt








In [36]:
#TODO mc = 0.1, 0.5, 1.0


In [15]:
#TODO???
# exclude/encode action: 1M, 5M
